# Chapter 28 – Worked examples

This notebook runs the examples of [Chapter 28](../index.md) live: the DC-OPF LP and the optimal transmission
switching MILP of the worked example on IEEE 30, the 3-bus case where switching lowers the cost, and ToOp's AC
acceptance rule. The code lives in the modules next to it (`ieee30_ots.py`, `braess3.py`, `acceptance.py`); the
cells only call them.

Run it from this folder (the kernel's working directory must be `src/`).

In [ ]:
import acceptance
import braess3
import ieee30_ots
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)

## DC-OPF as an LP

`ieee30_ots.dc_opf` writes the LP of the worked example with PuLP and solves it with HiGHS. The duals of the nodal
balance constraints are the locational marginal prices: equal everywhere without congestion, different on the two
sides of a line at its rating.

In [ ]:
case = ieee30_ots.data()
opf = ieee30_ots.dc_opf(case)
print(f'cost {opf["cost"]:.1f} $/h, dispatch {np.round(opf["p"], 1)} MW')
loading = 100 * np.abs(opf['f']) / case['rating']
print(
    'lines at their rating:',
    [f'{i + 1}-{j + 1}' for (i, j), load in zip(case['lines'], loading, strict=True) if load > 99.999],
)
print(f'LMP from {opf["lmp"].min():.2f} to {opf["lmp"].max():.2f} $/MWh')

## Optimal transmission switching

`ieee30_ots.ots` adds the binary $z_\ell$ and the big-M constraints of the worked example. Opening one line
lowers the cost; the brute force over the same sets gives the same optimum with 42 LPs.

In [ ]:
r = ieee30_ots.ots(case, 1, list(range(41)))
print(f'k = 1: cost {r["cost"]:.1f} $/h, switched {r["switched"]}, {r["time_s"]:.2f} s, gap {r["gap"]:.3g}')
best, arg, count, dt = ieee30_ots.brute_force(case, 1, list(range(41)))
print(f'brute force: {best:.1f} $/h with {arg}, {count} LPs in {dt:.1f} s')

## Switching off a line can lower the cost

In the 3-bus case of Exercise 2 the direct line 1–3 carries two thirds of the transfer and has the lowest rating.
Opening it frees the cheap generator; the LMPs show where the congestion rent went.

In [ ]:
for name, open_lines in (('closed', ()), ('line 1-3 open', (2,))):
    r = braess3.dc_opf(open_lines)
    print(f'{name:14s} cost {r["cost"]:.0f} $/h, dispatch {np.round(r["p"], 1)} MW, LMP {np.round(r["lmp"], 1)} $/MWh')

## The acceptance rule and its traps

`acceptance.evaluate_acceptance` is a port of ToOp's rule. Multiplicative thresholds against a zero baseline
reject any positive value, however small.

In [ ]:
print(
    'zero baseline:',
    acceptance.evaluate_acceptance(
        {'overload_energy_n_1': 0.01, 'critical_branch_count_n_1': 0},
        {'overload_energy_n_1': 0.0, 'critical_branch_count_n_1': 0},
    ),
)
print(
    '1 MW baseline:',
    acceptance.evaluate_acceptance(
        {'overload_energy_n_1': 0.01, 'critical_branch_count_n_1': 0},
        {'overload_energy_n_1': 1.0, 'critical_branch_count_n_1': 0},
    ),
)